In [ ]:
import torch
import random

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
# Takes all the data and creates an encoding map
def process(data):
  raw_chars = list(data)
  unique_chars = sorted(list(set(raw_chars)))
  encode_dic = {}
  decode_dic = {}

  for token_id, char in enumerate(unique_chars):
    encode_dic[char] = token_id
    decode_dic[token_id] = char

  return encode_dic, decode_dic, unique_chars
# Returns the string as a list of tokens
def encode(encoding_dic, x):
  chars = list(x)
  tokens = []
  for c in chars:
    tokens.append(encoding_dic[c])

  return tokens
# Turns a list of tokens into a list of chars
def decode(decoding_dic, tokens):
  chars = []
  for tok in tokens:
    chars.append(decoding_dic[tok])

  return chars



In [ ]:
# Creates embedding matrix of size CxC or vocab size x vocab size - weights initialized to 1/C
def embedding_matrix(C):
  matrix = torch.full((C, C), 1/C, device=device) + (torch.randn(C, C, device=device) * 0.01)
  return matrix


def convert_one_hot(idy, C):
  y_one_hot = torch.zeros(C, device=device)
  y_one_hot[idy] = 1
  return y_one_hot


In [ ]:
# Gets logits from embedding matrix and calculates soft maxes
def soft_max(idx, e_matrix):
  logits = e_matrix[idx].squeeze()

  numerators = logits.exp()       # Exponentiate everything at once (f(x) = e^x)
  denominators = numerators.sum() # Sum exponents
  ps = numerators/denominators

  return ps

# Compute gradient given the input - Returns CxC matrix to be used in weight update
def gradient(idx, idy, y_one_hot, e_matrix):
  # Get softmaxes
  ps = soft_max(idx, e_matrix)
  error = ps - y_one_hot

  # Loss - just for info
  loss = torch.log(ps[idy]) * -1

  # Simpler then X @ error.T -
  # Emptry gradient matrix
  grad = torch.zeros_like(e_matrix)

  # Update row 'idx'
  grad[idx] = error

  return grad, loss

# Updates weight matrix with learning rate specified
def update(grad, e_matrix, lr):
  e_matrix.sub_(lr * grad)




In [ ]:
# Run loop to continuously train bigram model
def train(T, data, encoding_dic, e_matrix, iter=1000, lr=1e-3):
  # T       - length of a sequence
  # data    - actual data we will be learning
  # enc_dic - dictionairy used for encoding
  # e_mat   - matrix representing weights
  # iter    - number of updates to be done
  # lr      - learning rate

  index_pool = len(data) - T - 1  # For choose random starting point and allow for size T sequence

  for i in range(iter):
    starting_i = random.randint(0, index_pool)
    batch = data[starting_i:starting_i + T + 1]

    # Decrease learning rate over time
    if i % 2000 == 0:
      lr *= 0.5

    losses = []

    # Create multiple x, y pairs for relevant updates
    for j in range(len(batch)-1):
      x = batch[0:j+1]
      y = batch[j+1]

      # For now only use last of x (attention not yet implemented)
      x = x[-1]
      idx = encode(encoding_dic, x)[0]
      idy = encode(encoding_dic, y)[0]
      y_one_hot = convert_one_hot(idy, len(vocab))

      # Compute loss and update weights
      grad, loss = gradient(idx, idy, y_one_hot, e_matrix)
      update(grad, e_matrix, lr)

      losses.append(loss)
    if i%5000 == 0:
      print(f"Loss on iteration {i}: {sum(losses) / len(losses)}\n")

  return e_matrix




In [ ]:
# Read data
with open("/tiny-shakespeare.txt", 'r') as f:
  data = f.read()

In [ ]:
# Get vocab and encoding/decoding dictionairies
encoding_dic, decoding_dic, vocab = process(data)

C = len(vocab)
e_matrix = embedding_matrix(C)

In [ ]:
# Generate token using existing weights
def generate(idx, e_matrix, decoding_dic, max_tokens=128):
  # idx     - starting point (input)
  # e_mat   - weights used for generation
  # dec_dic - dictionary used for decoding tokens

  generated_tokens = [idx]  # Used to generate and add to one array

  for i in range(max_tokens):
    x = generated_tokens[-1]
    soft_maxes = soft_max(x, e_matrix)

    # Multinomial sampling
    cdf = torch.cumsum(soft_maxes, dim=0)
    cdf[-1] = 1.0
    ran = torch.rand(1, device=device)
    next_idx = torch.where(ran < cdf)[0][0].item()

    generated_tokens.append(next_idx)

  return generated_tokens



In [ ]:
e_matrix = train(8, data, encoding_dic, e_matrix, 100000, 1e-1)


Loss on iteration 0_7: 2.7610273361206055

Loss on iteration 5000_7: 1.816498041152954

Loss on iteration 10000_7: 2.0514795780181885

Loss on iteration 15000_7: 1.63631010055542

Loss on iteration 20000_7: 2.6418704986572266

Loss on iteration 25000_7: 3.353717565536499

Loss on iteration 30000_7: 2.89385986328125

Loss on iteration 35000_7: 3.1955881118774414

Loss on iteration 40000_7: 1.252082109451294

Loss on iteration 45000_7: 1.252082109451294

Loss on iteration 50000_7: 3.17231822013855

Loss on iteration 55000_7: 2.217353582382202

Loss on iteration 60000_7: 4.329949855804443

Loss on iteration 65000_7: 2.3888537883758545

Loss on iteration 70000_7: 1.5372451543807983

Loss on iteration 75000_7: 1.252082109451294

Loss on iteration 80000_7: 1.6312657594680786

Loss on iteration 85000_7: 3.013876438140869

Loss on iteration 90000_7: 2.1136159896850586

Loss on iteration 95000_7: 2.701249122619629



In [ ]:
starting_token = "H"
starting_token = encode(encoding_dic, starting_token)[0]

generated_tokens = generate(starting_token, e_matrix, decoding_dic, 512)

generation = decode(decoding_dic, generated_tokens)
generation = "".join(generation)

print(generation)

Hest torinkLARUShes!
CI or slyle whetorr isofon utonal asi$P; agre
O. bee ulwe mintofr ouJUy byo theve me,
S:
Nand$VGor I mmy, se omayeacay,
S:
T:
AVINa
Wt RursIqVIO: enoubeatelit.
I susicGse hons wsis mur youth byprmey me
d s angores, twin, fo, itheranthorys!O:
Rmul ichamund alyokes fe?


INRYVI!
O&$.
CLUincLre'Pd s,
Th IN3$ ayo snd.
Ahtsautous?&fSeas they omern anncorty CELINomeas

Ff me sestha willy, ns tey h rin ISDXfowiereszameve thourrsisen aselFor s bmalot st dot ck s djLB
T: mHNCERhem aRO d las y ist
